In [0]:
from src.config import (
    TEST_TABLE,
    FEATURE_TABLE,
    REGISTERED_MODEL,
    EXPERIMENT_NAME,
    TARGET_COLUMN,
)

print("==========================================")
print("EVALUATION CONFIGURATION")
print("==========================================")

print("TEST TABLE      :", TEST_TABLE)
print("FEATURE TABLE   :", FEATURE_TABLE)
print("REGISTERED MODEL:", REGISTERED_MODEL)
print("EXPERIMENT      :", EXPERIMENT_NAME)
print("TARGET COLUMN   :", TARGET_COLUMN)

print("==========================================")

In [0]:
import mlflow
import mlflow.sklearn

import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

print(
    "Libraries imported successfully."
)

In [0]:
client = mlflow.MlflowClient()

print(
    "MLflow client created."
)

In [0]:
test_df = spark.table(
    TEST_TABLE
)

print(
    "Test table:",
    TEST_TABLE
)

print(
    "Test rows:",
    test_df.count()
)

display(test_df)

In [0]:
required_test_columns = [
    "record_id",
    "sepal_length",
    "sepal_width",
    "petal_length",
    "petal_width",
    "target",
    "species",
]

missing_columns = [
    column
    for column in required_test_columns
    if column not in test_df.columns
]

assert not missing_columns, (
    f"Missing test columns: {missing_columns}"
)

print(
    "Test schema validation: PASSED"
)

In [0]:
null_count = (
    test_df
    .select(
        required_test_columns
    )
    .dropna()
    .count()
)

total_count = test_df.count()

assert null_count == total_count, (
    "Test dataset contains null values."
)

duplicate_count = (
    test_df
    .dropDuplicates(
        ["record_id"]
    )
    .count()
)

assert duplicate_count == total_count, (
    "Duplicate record_id values found."
)

print(
    "Test data validation: PASSED"
)

In [0]:
from src.feature_engineering import (
    create_features,
    get_feature_columns,
)

feature_columns = get_feature_columns()

print(
    "Evaluation features:"
)

for column in feature_columns:
    print(
        " -",
        column
    )

In [0]:
test_feature_df = create_features(
    test_df
)

display(
    test_feature_df
)

In [0]:
from src.feature_engineering import (
    validate_features
)

validate_features(
    test_feature_df
)

print(
    "Test feature validation: PASSED"
)

In [0]:
test_pdf = (
    test_feature_df
    .select(
        *feature_columns,
        TARGET_COLUMN
    )
    .toPandas()
)

print(
    "Test DataFrame shape:",
    test_pdf.shape
)

display(
    test_pdf.head()
)

In [0]:
X_test = test_pdf[
    feature_columns
]

y_test = test_pdf[
    TARGET_COLUMN
]

print(
    "X_test shape:",
    X_test.shape
)

print(
    "y_test shape:",
    y_test.shape
)

In [0]:
assert len(X_test) == len(y_test)

assert X_test.isnull().sum().sum() == 0

assert y_test.isnull().sum() == 0

assert set(
    y_test.unique()
).issubset({0, 1, 2})

print(
    "Evaluation data validation: PASSED"
)

In [0]:
challenger = (
    client.get_model_version_by_alias(
        REGISTERED_MODEL,
        "Challenger"
    )
)

challenger_version = challenger.version

print("==========================================")
print("CHALLENGER MODEL")
print("==========================================")

print(
    "Model:",
    challenger.name
)

print(
    "Version:",
    challenger_version
)

print(
    "Alias: Challenger"
)

print("==========================================")

In [0]:
challenger_model_uri = (
    f"models:/{REGISTERED_MODEL}@Challenger"
)

print(
    "Loading model:"
)

print(
    challenger_model_uri
)

challenger_model = (
    mlflow.sklearn.load_model(
        challenger_model_uri
    )
)

print(
    "Challenger model loaded successfully."
)

In [0]:
challenger_predictions = (
    challenger_model.predict(
        X_test
    )
)

print(
    "Predictions generated:",
    len(challenger_predictions)
)

print(
    "First 10 predictions:"
)

print(
    challenger_predictions[:10]
)

In [0]:
challenger_accuracy = accuracy_score(
    y_test,
    challenger_predictions
)

challenger_precision = precision_score(
    y_test,
    challenger_predictions,
    average="weighted",
    zero_division=0,
)

challenger_recall = recall_score(
    y_test,
    challenger_predictions,
    average="weighted",
    zero_division=0,
)

challenger_f1 = f1_score(
    y_test,
    challenger_predictions,
    average="weighted",
    zero_division=0,
)

print("==========================================")
print("CHALLENGER TEST METRICS")
print("==========================================")

print(
    f"Accuracy : {challenger_accuracy:.4f}"
)

print(
    f"Precision: {challenger_precision:.4f}"
)

print(
    f"Recall   : {challenger_recall:.4f}"
)

print(
    f"F1 Score : {challenger_f1:.4f}"
)

print("==========================================")

In [0]:
challenger_cm = confusion_matrix(
    y_test,
    challenger_predictions
)

print(
    "Challenger Confusion Matrix:"
)

print(
    challenger_cm
)

In [0]:
print(
    classification_report(
        y_test,
        challenger_predictions,
        target_names=[
            "setosa",
            "versicolor",
            "virginica",
        ],
    )
)

In [0]:
challenger_metrics = {
    "model_version": int(
        challenger_version
    ),
    "accuracy": float(
        challenger_accuracy
    ),
    "precision": float(
        challenger_precision
    ),
    "recall": float(
        challenger_recall
    ),
    "f1_score": float(
        challenger_f1
    ),
}

challenger_metrics

In [0]:
champion_exists = True

try:

    champion = (
        client.get_model_version_by_alias(
            REGISTERED_MODEL,
            "Champion"
        )
    )

    champion_version = champion.version

    print(
        "Champion exists."
    )

    print(
        "Champion version:",
        champion_version
    )

except Exception as e:

    champion_exists = False
    champion = None
    champion_version = None

    print(
        "No Champion model currently exists."
    )

In [0]:
if not champion_exists:

    print(
        "No Champion exists."
    )

    print(
        "Challenger can become the initial Champion."
    )

In [0]:
if not champion_exists:

    client.set_registered_model_alias(
        name=REGISTERED_MODEL,
        alias="Champion",
        version=challenger_version,
    )

    print(
        f"Model version {challenger_version} "
        f"promoted to Champion."
    )

In [0]:
if not champion_exists:

    champion = (
        client.get_model_version_by_alias(
            REGISTERED_MODEL,
            "Champion"
        )
    )

    print(
        "Champion version:",
        champion.version
    )

In [0]:
if champion_exists:

    champion_model_uri = (
        f"models:/{REGISTERED_MODEL}@Champion"
    )

    champion_model = (
        mlflow.sklearn.load_model(
            champion_model_uri
        )
    )

    champion_predictions = (
        champion_model.predict(
            X_test
        )
    )

    champion_accuracy = accuracy_score(
        y_test,
        champion_predictions
    )

    champion_precision = precision_score(
        y_test,
        champion_predictions,
        average="weighted",
        zero_division=0,
    )

    champion_recall = recall_score(
        y_test,
        champion_predictions,
        average="weighted",
        zero_division=0,
    )

    champion_f1 = f1_score(
        y_test,
        champion_predictions,
        average="weighted",
        zero_division=0,
    )

    print("==========================================")
    print("CHAMPION TEST METRICS")
    print("==========================================")

    print(
        f"Champion Version: {champion_version}"
    )

    print(
        f"Accuracy : {champion_accuracy:.4f}"
    )

    print(
        f"Precision: {champion_precision:.4f}"
    )

    print(
        f"Recall   : {champion_recall:.4f}"
    )

    print(
        f"F1 Score : {champion_f1:.4f}"
    )

    print("==========================================")

In [0]:
if champion_exists:

    print("==========================================")
    print("CHAMPION vs CHALLENGER")
    print("==========================================")

    print(
        f"Champion   F1: {champion_f1:.4f}"
    )

    print(
        f"Challenger F1: {challenger_f1:.4f}"
    )

    if challenger_f1 > champion_f1:

        challenger_is_better = True

        print(
            "RESULT: Challenger is better."
        )

    else:

        challenger_is_better = False

        print(
            "RESULT: Champion remains better "
            "or equal."
        )

    print("==========================================")

In [0]:
if champion_exists and challenger_is_better:

    client.set_registered_model_alias(
        name=REGISTERED_MODEL,
        alias="Champion",
        version=challenger_version,
    )

    print(
        f"Version {challenger_version} "
        f"promoted to Champion."
    )

elif champion_exists:

    print(
        f"Version {champion_version} "
        f"remains Champion."
    )

else:

    print(
        f"Version {challenger_version} "
        f"is the initial Champion."
    )

In [0]:
evaluation_summary = {
    "challenger_version": int(
        challenger_version
    ),
    "challenger_accuracy": float(
        challenger_accuracy
    ),
    "challenger_precision": float(
        challenger_precision
    ),
    "challenger_recall": float(
        challenger_recall
    ),
    "challenger_f1": float(
        challenger_f1
    ),
    "champion_exists_before_evaluation": (
        champion_exists
    ),
}

if champion_exists:

    evaluation_summary[
        "champion_version"
    ] = int(champion_version)

    evaluation_summary[
        "champion_accuracy"
    ] = float(champion_accuracy)

    evaluation_summary[
        "champion_precision"
    ] = float(champion_precision)

    evaluation_summary[
        "champion_recall"
    ] = float(champion_recall)

    evaluation_summary[
        "champion_f1"
    ] = float(champion_f1)

    evaluation_summary[
        "challenger_promoted"
    ] = bool(
        challenger_is_better
    )

else:

    evaluation_summary[
        "champion_version"
    ] = int(challenger_version)

    evaluation_summary[
        "challenger_promoted"
    ] = True


evaluation_summary

In [0]:
with mlflow.start_run(
    run_name="iris_model_evaluation"
) as evaluation_run:

    mlflow.log_params(
        {
            "registered_model": REGISTERED_MODEL,
            "challenger_version": challenger_version,
        }
    )

    mlflow.log_metrics(
        {
            "challenger_accuracy": challenger_accuracy,
            "challenger_precision": challenger_precision,
            "challenger_recall": challenger_recall,
            "challenger_f1": challenger_f1,
        }
    )

    if champion_exists:

        mlflow.log_params(
            {
                "champion_version": champion_version,
            }
        )

        mlflow.log_metrics(
            {
                "champion_accuracy": champion_accuracy,
                "champion_precision": champion_precision,
                "champion_recall": champion_recall,
                "champion_f1": champion_f1,
            }
        )

    mlflow.set_tags(
        {
            "project": "iris-mlops",
            "evaluation_type": "model_comparison",
            "dataset": "iris_test",
        }
    )

    evaluation_run_id = (
        evaluation_run.info.run_id
    )

print(
    "Evaluation MLflow Run ID:",
    evaluation_run_id
)

In [0]:
final_champion = (
    client.get_model_version_by_alias(
        REGISTERED_MODEL,
        "Champion"
    )
)

print("==========================================")
print("FINAL MODEL ALIASES")
print("==========================================")

print(
    "Champion version:",
    final_champion.version
)

print(
    "Challenger version:",
    challenger_version
)

print("==========================================")

In [0]:
assert final_champion is not None

assert challenger_version is not None

assert challenger_accuracy >= 0.0
assert challenger_accuracy <= 1.0

assert challenger_f1 >= 0.0
assert challenger_f1 <= 1.0

print("==========================================")
print("04_EVALUATE COMPLETED SUCCESSFULLY")
print("==========================================")

print(
    f"Challenger Version : {challenger_version}"
)

print(
    f"Challenger Accuracy : "
    f"{challenger_accuracy:.4f}"
)

print(
    f"Challenger F1       : "
    f"{challenger_f1:.4f}"
)

print(
    f"Champion Version    : "
    f"{final_champion.version}"
)

print("==========================================")